In [1]:
# Instalar librerías necesarias
!pip install opencv-python

# Taller Práctico 2 – Procesamiento de Video por Frames
Arquitectura usada en clase: **Video → Frames → Aplicar filtro → Reconstruir video**.

Juan Pablo Beltran Santana

In [2]:
# Subir el video
from google.colab import files
uploaded = files.upload()

Saving Above the clouds sunrise timelapse HD   Amanecer por encima de las nubes HD.mp4 to Above the clouds sunrise timelapse HD   Amanecer por encima de las nubes HD.mp4


In [3]:
# Verificar nombre del archivo
import os
os.listdir()

['.config',
 'Above the clouds sunrise timelapse HD   Amanecer por encima de las nubes HD.mp4',
 'sample_data']

Este código abre el video usando OpenCV y lo divide en frames (imágenes). Cada frame se guarda automáticamente en una carpeta llamada frames con un nombre numerado para mantener el orden. El proceso continúa hasta llegar al final del video y al final se muestra la cantidad total de frames extraídos.

In [5]:
# Separar video en frames
import cv2
import os

video_path = 'Above the clouds sunrise timelapse HD   Amanecer por encima de las nubes HD.mp4'

frames_dir = 'frames'
os.makedirs(frames_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)

frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    cv2.imwrite(f"{frames_dir}/frame_{frame_count:05d}.png", frame)
    frame_count += 1

cap.release()

print('Frames extraidos:', frame_count)

Frames extraidos: 273


In [6]:
# Verificar cuantos frames se generaron
import os
print('Total frames:', len(os.listdir('frames')))

Total frames: 273


## Filtro 1 – Negativo

In [7]:
import cv2
import os

output_dir = 'frames_negativo'
os.makedirs(output_dir, exist_ok=True)

for file in sorted(os.listdir('frames')):
    img = cv2.imread(os.path.join('frames', file))
    negativo = 255 - img
    cv2.imwrite(os.path.join(output_dir, file), negativo)

print('Filtro negativo aplicado')

Filtro negativo aplicado


## Filtro 2 – Binario

In [8]:
import cv2
import os

output_dir = 'frames_binario'
os.makedirs(output_dir, exist_ok=True)

threshold = 128

for file in sorted(os.listdir('frames')):
    img = cv2.imread(os.path.join('frames', file))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    cv2.imwrite(os.path.join(output_dir, file), binary)

print('Filtro binario aplicado')

Filtro binario aplicado


## Filtro 3 – Blur

In [9]:
import cv2
import os

output_dir = 'frames_blur'
os.makedirs(output_dir, exist_ok=True)

for file in sorted(os.listdir('frames')):
    img = cv2.imread(os.path.join('frames', file))
    blur = cv2.blur(img, (3,3))
    cv2.imwrite(os.path.join(output_dir, file), blur)

print('Filtro blur aplicado')

Filtro blur aplicado


## Filtro 4 – Recorte

In [10]:
import cv2
import os

output_dir = 'frames_crop'
os.makedirs(output_dir, exist_ok=True)

x_start = 100
y_start = 100
width = 300
height = 300

for file in sorted(os.listdir('frames')):
    img = cv2.imread(os.path.join('frames', file))
    crop = img[y_start:y_start+height, x_start:x_start+width]
    cv2.imwrite(os.path.join(output_dir, file), crop)

print('Recorte aplicado')

Recorte aplicado


## Filtro 5 – Sobel

In [11]:
import cv2
import os
import numpy as np

output_dir = 'frames_sobel'
os.makedirs(output_dir, exist_ok=True)

for file in sorted(os.listdir('frames')):
    img = cv2.imread(os.path.join('frames', file))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

    sobel = np.sqrt(sobelx**2 + sobely**2)
    sobel = np.uint8(np.clip(sobel,0,255))

    cv2.imwrite(os.path.join(output_dir, file), sobel)

print('Filtro Sobel aplicado')

Filtro Sobel aplicado


## Reconstruir el video (ejemplo con filtro negativo)
En esta parte es solo para el filtro negativo.

In [12]:
import cv2
import os

frame_folder = 'frames_negativo'

frame_files = sorted(os.listdir(frame_folder))

first_frame = cv2.imread(os.path.join(frame_folder, frame_files[0]))

height, width, layers = first_frame.shape

video = cv2.VideoWriter(
    'video_negativo.mp4',
    cv2.VideoWriter_fourcc(*'mp4v'),
    30,
    (width, height)
)

for file in frame_files:
    img = cv2.imread(os.path.join(frame_folder, file))
    video.write(img)

video.release()

print('Video creado')

Video creado


Resultado para descargar

In [17]:
# Descargar video resultado
from google.colab import files
files.download('video_negativo.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Construimos los 5 filtros

In [24]:
import cv2
import os

def reconstruir_video(frame_folder, output_name, fps=30):

    frame_files = sorted(os.listdir(frame_folder))

    if len(frame_files) == 0:
        print("No hay frames en:", frame_folder)
        return

    first_frame = cv2.imread(os.path.join(frame_folder, frame_files[0]))

    height, width, layers = first_frame.shape

    video = cv2.VideoWriter(
        output_name,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (width, height)
    )

    for file in frame_files:

        img_path = os.path.join(frame_folder, file)
        img = cv2.imread(img_path)

        if img is None:
            continue

        video.write(img)

    video.release()

    print("Video creado:", output_name)

Recontruimos cada video con si respectivo filtro.

In [25]:
reconstruir_video("frames_negativo", "video_negativo.mp4")

reconstruir_video("frames_binario", "video_binario.mp4")

reconstruir_video("frames_blur", "video_blur.mp4")

reconstruir_video("frames_crop", "video_crop.mp4")

reconstruir_video("frames_sobel", "video_sobel.mp4")

Video creado: video_negativo.mp4
Video creado: video_binario.mp4
Video creado: video_blur.mp4
Video creado: video_crop.mp4
Video creado: video_sobel.mp4


Acá puedes descargar los 5 filtros con el mismo video.

In [26]:
from google.colab import files

files.download("video_negativo.mp4")
files.download("video_binario.mp4")
files.download("video_blur.mp4")
files.download("video_crop.mp4")
files.download("video_sobel.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>